In [1]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\abhay\\Downloads\\Medical-Chatbot-Generative-AI'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
! pip install --upgrade sentence-transformers


In [6]:
from pinecone import Pinecone

c:\Users\abhay\.conda\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
! pip install langchain-groq


In [8]:
# -----------------------
# 0. Import necessary modules
# -----------------------
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set your API Keys
GROQ_API_KEY = "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"
PINECONE_API_KEY = "pcsk_7L4dwj_5zyQBcw6KboyFBSGnEbfYhH79CCKcN6wYgfAAfTnVb1sHNdiuj3kVsSgDvvHafr"

# Set environment variables
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# -----------------------
# 1. Load and Split PDF Data
# -----------------------
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_pdf_file(data_path):
    loader = DirectoryLoader(data_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

# Load the PDFs
extracted_data = load_pdf_file(data_path="Data/")

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = text_split(extracted_data)
print("Length of Text Chunks:", len(text_chunks))

# -----------------------
# 2. Download HuggingFace Embeddings
# -----------------------
from langchain.embeddings import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

embeddings = download_hugging_face_embeddings()

query_result = embeddings.embed_query("Hello world")
print("Embedding vector length:", len(query_result))

# -----------------------
# 3. Set up Pinecone (LATEST 2025 METHOD)
# -----------------------
from pinecone import Pinecone, ServerlessSpec

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "mkc"

# Create index if it doesn't exist
'''if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
'''

# -----------------------
# 4. Upload Embeddings to Pinecone
# -----------------------
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

# Or connect to existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

# -----------------------
# 5. Set up Retriever
# -----------------------
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("What is Acne?")
print("Sample Retrieved Docs:", retrieved_docs)

# -----------------------
# 6. Set up Groq LLM (instead of OpenAI)
# -----------------------


Length of Text Chunks: 0


C:\Users\abhay\AppData\Local\Temp\ipykernel_2592\1542925495.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Embedding vector length: 384
Sample Retrieved Docs: []


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0.4,
    max_tokens=500,
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

# -----------------------
# 7. Create RAG Chain
# -----------------------
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# -----------------------
# 8. Test the RAG System
# -----------------------
response = rag_chain.invoke({"input": "Lately, I've been feeling extremely tired, even after a full night's sleep. I also noticed that my heart races sometimes and I get short of breath just from climbing a few stairs. I've had some swelling in my ankles too. I thought it was just stress, but it’s been going on for weeks now."})
print("Response 1:", response["answer"])



Response 1: It's possible that you may be experiencing symptoms related to a underlying medical condition, such as heart disease or anemia. I would recommend consulting a doctor to determine the cause of your fatigue, shortness of breath, and swelling. A medical professional can evaluate your symptoms and provide a proper diagnosis and treatment plan.


In [10]:
response = rag_chain.invoke({"input": "I have asthma and sometimes struggle with shortness of breath. What treatments are recommended, and what precautions should I take to manage my condition?"})
print("Response 1:", response["answer"])

Response 1: Recommended treatments for asthma include inhalers, such as bronchodilators and corticosteroids, as well as oral medications. To manage your condition, it's essential to work with your doctor to develop a personalized treatment plan and follow precautions such as avoiding triggers like pollen, dust, and smoke. Additionally, carrying a rescue inhaler and monitoring your symptoms regularly can help prevent asthma attacks and alleviate shortness of breath.


In [11]:
response = rag_chain.invoke({"input": "i am suffering from coldaswell as high pain in body suggest me some   medicine "})
print("Response 1:", response["answer"])

Response 1: I'm not a doctor, but for common cold and body pain, over-the-counter medications like paracetamol or acetaminophen (e.g., Tylenol) and ibuprofen (e.g., Advil) may help. However, it's recommended to consult a doctor or pharmacist for proper diagnosis and medication advice. I suggest you visit a healthcare professional for personalized guidance on treating your cold and body pain.


In [12]:
response = rag_chain.invoke({"input": " give me sybptom of maleria"})
print("Response 1:", response["answer"])

Response 1: Symptoms of malaria include fever, chills, flu-like symptoms, and in severe cases, coma or death. Other common symptoms are headache, muscle and joint pain, nausea, and vomiting. If left untreated, malaria can lead to severe anemia, respiratory distress, and organ failure.
